<a href="https://colab.research.google.com/github/Eng7ouda06/flyrank-ml-internship/blob/main/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Eng7ouda06/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Refresh / Content Opportunity Scoring**

I'm picking this lane because I already have direct evidence it's worth pursuing — Notebooks 1 and 2
showed a learned model beating a hand-written "stale x visible" rule by roughly 3x on Precision@50
(0.240 -> 0.740) on the exact same kind of refresh-review problem this lane targets. I also found that
naive intuitions (high search volume = more traffic, longer content = better trend) don't hold up in
this data, which tells me the useful signal here is not something a simple hand rule can capture -- it's
exactly the kind of compound pattern a model is built to find. This lane lets me build directly on work
I've already validated instead of starting a new investigation cold.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Question:** Given a page's current search and engagement signals, which pages should a content
reviewer look at first for refresh, expansion, or protection?

**Decision it improves:** which of thousands of pages a content/SEO reviewer spends their limited
time on first, out of a backlog too large to review manually.

**Who acts on it:** a content strategist or SEO reviewer with a fixed weekly review capacity
(e.g. ~50 pages/week), who currently uses ad-hoc judgment or a simple hand rule to pick what to look at.

**Cost of a wrong call:**
- False positive (flagged as high-priority, isn't actually declining): wastes a reviewer's limited
  time on a page that didn't need attention, at the expense of a page that did.
- False negative (a genuinely declining page ranked low): the page keeps losing visibility
  undetected until the next review cycle, compounding lost traffic.
Given limited reviewer capacity, the real cost is opportunity cost, not catastrophic failure --
which is why this is a ranking/prioritization problem, not a high-stakes prediction problem.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*


Numbers pulled from the starter dataset (data/raw/content_refresh_anonymized.csv, ~30k pages)
and the pipeline's own verified output (outputs/model_results.json), computed live below.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Eng7ouda06/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"

# Run the pipeline so outputs/model_results.json exists in this session
subprocess.run([sys.executable, "scripts/run_all.py"], check=True)
print("Pipeline run complete. Ready.")


import pandas as pd, json

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
res = json.load(open("outputs/model_results.json"))

base_p50 = res["baseline"]["baseline_precision_at_50"]
rf_p50 = res["models"]["random_forest"]["precision_at_50"]
corr = df["search_volume"].corr(df["impressions_90d"])
wc = df.groupby("trend_direction")["word_count"].median()

print(f"Rows in starter dataset: {df.shape[0]}")
print(f"Baseline rule Precision@50: {base_p50:.3f}  ->  Random forest Precision@50: {rf_p50:.3f}  ({rf_p50/base_p50:.1f}x)")
print(f"Correlation(search_volume, impressions_90d): {corr:.3f}  <- near zero, so volume alone won't get us there")
print(f"Median word_count, down vs up pages: {wc['down']:.0f} vs {wc['up']:.0f}  <- nearly identical, length isn't the lever")

Working dir: /content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship
Pipeline run complete. Ready.
Rows in starter dataset: 30000
Baseline rule Precision@50: 0.240  ->  Random forest Precision@50: 0.740  (3.1x)
Correlation(search_volume, impressions_90d): 0.001  <- near zero, so volume alone won't get us there
Median word_count, down vs up pages: 2909 vs 2848  <- nearly identical, length isn't the lever


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:** observed associations between measurable signals (staleness, visibility,
position, CTR) and a page's recent trend direction; a decision-support ranking that helps a
reviewer prioritize a backlog; directional evidence that a learned combination of signals
outperforms a simple hand rule at this task, on this dataset.

**What I cannot claim:** that refreshing a page will cause it to recover (that requires an actual
experiment, not this data); that I've found a Google ranking factor; that any single page's decline
is fully "explained" by these signals; that results on this 30k-row starter slice will hold
identically on the full ~79M-row warehouse without re-validating.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.